# Super Stock Prediction Algorithm 
Stacked ensemble (RF + XGBoost + DNN => meta-learner) for both next-day **closing price** (regression) and **direction** (classification).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_path = "/kaggle/input/stock-market-data/stock_market_data/sp500/csv"

files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
print("Total CSV files found:", len(files))

df_list = []
for file in files:
    temp = pd.read_csv(os.path.join(data_path, file))
    temp['Ticker'] = file.replace('.csv', '')
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
print("Combined dataset shape:", df.shape)
df.head()


In [ ]:
# Basic cleanup
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date', 'Close'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print("After cleanup:", df.shape)


## Feature Engineering
All rolling features are computed **per ticker** using `groupby + apply` so rolling windows never bleed across different companies.


In [ ]:
def add_features(data):
    """Computes all temporal features for a single ticker."""
    data = data.copy()
    data['Returns']     = data['Close'].pct_change()
    data['LogReturns']  = np.log(data['Close'] / data['Close'].shift(1))
    data['MA_7']        = data['Close'].rolling(window=7).mean()
    data['MA_21']       = data['Close'].rolling(window=21).mean()
    data['MA_30']       = data['Close'].rolling(window=30).mean()
    data['Volatility_7']= data['Returns'].rolling(window=7).std()
    data['Volatility_21']= data['Returns'].rolling(window=21).std()
    data['Momentum']    = data['Close'] - data['Close'].shift(5)
    # Fix any inf values from log
    data['LogReturns']  = data['LogReturns'].replace([np.inf, -np.inf], np.nan)
    return data

# Appling per ticker as iy prevents cross-ticker rolling contamination
df = df.groupby('Ticker', group_keys=False).apply(add_features)
df = df.dropna().reset_index(drop=True)
print("After feature engineering:", df.shape)
df.head()


In [ ]:
sample_ticker = 'AAPL'
apple = df[df['Ticker'] == sample_ticker]

plt.figure(figsize=(12, 4))
plt.plot(apple['Date'], apple['Close'], label='Close')
plt.plot(apple['Date'], apple['MA_21'], label='MA_21', linestyle='--')
plt.title(f"{sample_ticker} Close vs MA_21")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Targets & Train/Test Split
> **Note:** `Close` (today's price) is included as a feature. This makes the regression task easier because tomorrow's close is usually close to today's.


In [ ]:
# Targets
df['Target']    = df.groupby('Ticker')['Close'].shift(-1)   # next-day close
df['Direction'] = (df['Target'] > df['Close']).astype(int)  # 1 = up, 0 = down
df = df.dropna(subset=['Target']).reset_index(drop=True)

FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Returns', 'LogReturns',
    'MA_7', 'MA_21', 'MA_30',
    'Volatility_7', 'Volatility_21', 'Momentum'
]

X = df[FEATURES].values
y_reg = df['Target'].values
y_cls = df['Direction'].values

# Chronological split
split = int(len(X) * 0.8)
X_train_raw, X_test_raw = X[:split], X[split:]
y_train_reg, y_test_reg = y_reg[:split], y_reg[split:]
y_train_cls, y_test_cls = y_cls[:split], y_cls[split:]

print(f"Train size: {len(X_train_raw):,}  |  Test size: {len(X_test_raw):,}")


## Scaling



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)   # fit+transform on TRAIN only
X_test  = scaler.transform(X_test_raw)         # transform ONLY on TEST

print("Scaler fitted on train set only — no test leakage.")
print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBRegressor, XGBClassifier
import numpy as np

print("YAY IMPORTS ARE OK.")


## Part 1  Regression (Next-Day Close Price)


In [ ]:
#Base model definitions
rf_reg = RandomForestRegressor(
    n_estimators=500, max_depth=10,
    min_samples_split=5, min_samples_leaf=2,
    max_features='sqrt', n_jobs=-1, random_state=42
)
xgb_reg = XGBRegressor(
    n_estimators=800, learning_rate=0.03, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, tree_method='hist'
)

dnn_reg = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')
])
dnn_reg.compile(optimizer='adam', loss='mse')

print("Regression base models defined.")


In [ ]:
# Training base regression models
rf_reg.fit(X_train, y_train_reg)
print("RF regressor trained.")

xgb_reg.fit(X_train, y_train_reg)
print("XGB regressor trained.")

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
dnn_reg.fit(
    X_train, y_train_reg,
    epochs=50, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
print("DNN regressor trained.")


### Regression Meta-Learner
Base model predictions on **train** data are used to fit the meta-learner.
Base model predictions on **test** data are used for evaluation only, no leakage.


In [ ]:
# Base predictions on train (for fitting meta-learner)
rf_train_pred  = rf_reg.predict(X_train)
xgb_train_pred = xgb_reg.predict(X_train)
dnn_train_pred = dnn_reg.predict(X_train).flatten()
X_meta_train_reg = np.column_stack((rf_train_pred, xgb_train_pred, dnn_train_pred))

# Base predictions on test (for final evaluation)
rf_test_pred  = rf_reg.predict(X_test)
xgb_test_pred = xgb_reg.predict(X_test)
dnn_test_pred = dnn_reg.predict(X_test).flatten()
X_meta_test_reg = np.column_stack((rf_test_pred, xgb_test_pred, dnn_test_pred))

# Fiting metalearner on train metafeatures
meta_reg = RidgeCV(alphas=[0.1, 1.0, 10.0])
meta_reg.fit(X_meta_train_reg, y_train_reg)

# Evaluating on test meta-features
meta_preds_reg = meta_reg.predict(X_meta_test_reg)

mae  = mean_absolute_error(y_test_reg, meta_preds_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, meta_preds_reg))
r2   = r2_score(y_test_reg, meta_preds_reg)

print(f"Super Ensemble MAE:  {mae:.4f}")
print(f"Super Ensemble RMSE: {rmse:.4f}")
print(f"Super Ensemble: {r2:.4f}")


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test_reg[:500], label='Actual', alpha=0.8)
plt.plot(meta_preds_reg[:500], label='SSPA Predicted', alpha=0.8)
plt.legend()
plt.title("SSPA, Next Day Close Price, first 500 test samples)")
plt.tight_layout()
plt.show()


## Part 2 Classification (Next-Day Direction)


In [ ]:
rf_cls = RandomForestClassifier(
    n_estimators=300, max_depth=10, random_state=42, n_jobs=-1
)
xgb_cls = XGBClassifier(
    n_estimators=500, learning_rate=0.03, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, tree_method='hist'
)
dnn_cls = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
dnn_cls.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Classification base models defined.")


In [ ]:
rf_cls.fit(X_train, y_train_cls)
print("RF classifier trained.")

xgb_cls.fit(X_train, y_train_cls)
print("XGB classifier trained.")

early_stop_cls = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
dnn_cls.fit(
    X_train, y_train_cls,
    epochs=30, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop_cls],
    verbose=1
)
print("DNN classifier trained.")


### Classification Meta-Learner 

In [ ]:
# Base probabilities on train (for fitting meta-learner)
rf_train_proba  = rf_cls.predict_proba(X_train)[:, 1]
xgb_train_proba = xgb_cls.predict_proba(X_train)[:, 1]
dnn_train_proba = dnn_cls.predict(X_train).flatten()
X_meta_train_cls = np.column_stack((rf_train_proba, xgb_train_proba, dnn_train_proba))

# Base probabilities on tes (for evaluation only)
rf_test_proba  = rf_cls.predict_proba(X_test)[:, 1]
xgb_test_proba = xgb_cls.predict_proba(X_test)[:, 1]
dnn_test_proba = dnn_cls.predict(X_test).flatten()
X_meta_test_cls = np.column_stack((rf_test_proba, xgb_test_proba, dnn_test_proba))

# Fit meta-learner on train meta-features
meta_cls = LogisticRegression(max_iter=500)
meta_cls.fit(X_meta_train_cls, y_train_cls)

# Evaluate on test meta-features, only held-out data is used here
final_proba_cls  = meta_cls.predict_proba(X_meta_test_cls)[:, 1]
final_preds_cls  = (final_proba_cls > 0.5).astype(int)

acc = accuracy_score(y_test_cls, final_preds_cls)
auc = roc_auc_score(y_test_cls, final_proba_cls)
cm  = confusion_matrix(y_test_cls, final_preds_cls)

print(f"Direction accuracy {acc:.3f}")
print(f"Direction auc {auc:.3f}")
print("Confusion matrix:")
print(cm)


## Inference, Predict on New Data


In [ ]:
import yfinance as yf

def get_latest_stock_data(symbol="AAPL", period="3mo", interval="1d"):
    data = yf.download(symbol, period=period, interval=interval)
    data.reset_index(inplace=True)
    return data

def preprocess_new_data(raw_df):
    """Apply the same feature engineering used during training."""
    df = raw_df.copy()
    df['Returns']      = df['Close'].pct_change()
    df['LogReturns']   = np.log(df['Close'] / df['Close'].shift(1))
    df['MA_7']         = df['Close'].rolling(window=7).mean()
    df['MA_21']        = df['Close'].rolling(window=21).mean()
    df['MA_30']        = df['Close'].rolling(window=30).mean()
    df['Volatility_7'] = df['Returns'].rolling(window=7).std()
    df['Volatility_21']= df['Returns'].rolling(window=21).std()
    df['Momentum']     = df['Close'] - df['Close'].shift(5)
    df['LogReturns']   = df['LogReturns'].replace([np.inf, -np.inf], np.nan)
    df = df.dropna()
    return df

def predict_next_day(symbol, scaler, rf_reg, xgb_reg, dnn_reg, meta_reg,
                                    rf_cls, xgb_cls, dnn_cls, meta_cls):
    raw = get_latest_stock_data(symbol)
    processed = preprocess_new_data(raw)

    features = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'Returns', 'LogReturns',
        'MA_7', 'MA_21', 'MA_30',
        'Volatility_7', 'Volatility_21', 'Momentum'
    ]
    X_new = scaler.transform(processed[features].values)  #use fitted scaler

    #Regression
    rf_p  = rf_reg.predict(X_new)
    xgb_p = xgb_reg.predict(X_new)
    dnn_p = dnn_reg.predict(X_new).flatten()
    meta_input_reg = np.column_stack((rf_p, xgb_p, dnn_p))
    predicted_close = meta_reg.predict(meta_input_reg)[-1]

    #Classification
    rf_prob  = rf_cls.predict_proba(X_new)[:, 1]
    xgb_prob = xgb_cls.predict_proba(X_new)[:, 1]
    dnn_prob = dnn_cls.predict(X_new).flatten()
    meta_input_cls = np.column_stack((rf_prob, xgb_prob, dnn_prob))
    direction_prob = meta_cls.predict_proba(meta_input_cls)[-1, 1]
    direction = "Up" if direction_prob > 0.5 else "Down"

    return predicted_close, direction, direction_prob


In [ ]:
#Run inference
close, direction, prob = predict_next_day(
    "AAPL", scaler,
    rf_reg, xgb_reg, dnn_reg, meta_reg,
    rf_cls, xgb_cls, dnn_cls, meta_cls
)
print(f"Predicted next closing price ${close:.2f}")
print(f"Predicted direction {direction}  (confidence: {prob:.1%})")


## Streamlit App
The code below is a **standalone script**,
 save it as `streamlit_app.py` and run with `streamlit run streamlit_app.py`.
It cannot run inside a Jupyter/Kaggle notebook cell.


In [ ]:
#Save this as streamlit_app.py and run: streamlit run streamlit_app.py
#(Do NOT execute this cell in Jupyter/Kaggle)
"""
import streamlit as st
# Assumes trained models (rf_reg, xgb_reg, dnn_reg, meta_reg,
#                          rf_cls, xgb_cls, dnn_cls, meta_cls, scaler)
# are saved via joblib/pickle and loaded here.

st.title("Super stock predctor)")
symbol = st.text_input("Enter Stock Symbol", "AAPL")

if st.button("Predict Next Day"):
    close, direction, prob = predict_next_day(
        symbol, scaler,
        rf_reg, xgb_reg, dnn_reg, meta_reg,
        rf_cls, xgb_cls, dnn_cls, meta_cls
    )
    st.success(f"Predicted close: ${close:.2f}")
    st.info(f"Direction: {direction}  (confidence: {prob:.1%})")
"""
print("Streamlit snippet — run as a standalone script, not here.")


## Planned Improvements
- RSI, MACD, Bollinger Bands, and other technical indicators
- Finnhub API for live/real-time data
- Walk-forward cross-validation for more robust evaluation
- Optuna hyperparameter search
